# E-Commerce Anomaly Detection — Live Demo

**Architecture:** TGAT (Temporal Graph Attention) + E10 CatBoost Ensemble  
**Causal Protocol:** 7-day label availability boundary  
**Authoritative Test F1:** 62.25% (E10, frozen)

This notebook clones the repository, loads both frozen models (TGAT + E10), starts FastAPI, and creates a public demo via ngrok.

In [ ]:
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/sanjais146/anomoly_detection"

def run_cmd(cmd, check=False):
    print(f"> {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  stderr: {result.stderr[:500]}")
    else:
        if result.stdout.strip():
            print(f"  {result.stdout.strip()[:200]}")
    return result.returncode == 0

def bootstrap():
    print("Step 1: Bootstrapping Repository from GitHub...")
    repo_name = REPO_URL.rstrip('/').split('/')[-1]
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]

    if os.path.exists(repo_name):
        if not os.path.exists(f"{repo_name}/app") or not os.path.exists(f"{repo_name}/models"):
            print(f"  Removing corrupted directory '{repo_name}'...")
            shutil.rmtree(repo_name, ignore_errors=True)

    if not os.path.exists(repo_name) and not os.path.exists('app') and not os.path.exists('models'):
        print(f"  Cloning {REPO_URL}...")
        run_cmd(f"git clone {REPO_URL}")

    repo_root = None
    for root, dirs, files in os.walk('/content'):
        if 'app' in dirs and 'models' in dirs and 'frontend' in dirs:
            repo_root = root
            break

    if not repo_root and os.path.exists('app') and os.path.exists('models'):
        repo_root = os.getcwd()

    if repo_root:
        print(f"\u2705 Project root: {repo_root}")
        os.chdir(repo_root)
        sys.path.insert(0, repo_root)
        print("  Pulling Git LFS model files...")
        run_cmd("git lfs install")
        run_cmd("git lfs pull")
        return True
    else:
        print("\u274c ERROR: Could not find project root after clone.")
        return False

if not bootstrap():
    raise RuntimeError("Repository bootstrap failed.")

In [ ]:
def verify_models():
    print("Step 2: Verifying E10 and TGAT model artifacts...")
    required = {
        'models/e10_base.cbm':          100,
        'models/e10_deep.cbm':          100,
        'models/e10_weight.cbm':        100,
        'src/models/best_tgat_final.pt': 0.04,
        'app/main.py':                   0,
        'app/predictor.py':              0,
        'app/tgat_predictor.py':         0,
    }
    ok = True
    for f, min_mb in required.items():
        if not os.path.isfile(f):
            print(f"\u274c Missing: {f}")
            ok = False
        elif min_mb > 0 and os.path.getsize(f) / 1e6 < min_mb:
            size = os.path.getsize(f) / 1e6
            print(f"\u274c {f} too small ({size:.2f}MB < {min_mb}MB) — LFS pull failed?")
            ok = False
        else:
            size = os.path.getsize(f) / 1e6
            print(f"\u2705 {f} ({size:.2f} MB)")
    return ok

if not verify_models():
    raise RuntimeError("Model verification failed. Check Git LFS pull.")

In [ ]:
print("Step 3: Installing dependencies...")
!pip install -q fastapi uvicorn catboost pandas numpy pydantic requests pyngrok torch
print("\u2705 Dependencies installed.")

In [ ]:
import threading
import time
import requests
import uvicorn
import json
from app.main import app
from pyngrok import ngrok

print("Step 4: Authenticating ngrok...")
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    print("\u274c NGROK_AUTHTOKEN not found.")
    print("  1. Click the key icon in the Colab sidebar.")
    print("  2. Add secret: Name=NGROK_AUTHTOKEN, Value=<your_token>")
    print("  3. Enable 'Notebook access', then re-run this cell.")
    print("  Get your free token at: https://dashboard.ngrok.com")
    # Don't crash — continue so next cells show the issue
else:
    print("\u2705 NGROK_AUTHTOKEN found.")

print("\nStep 5: Starting FastAPI (TGAT + E10 Ensemble)...")
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("Step 6: Waiting for API health check...")
for i in range(20):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            health = r.json()
            e10_ok = health['components']['e10_catboost_ensemble']['loaded']
            tgat_ok = health['components']['tgat']['loaded']
            print(f"\u2705 FastAPI ONLINE — E10: {'OK' if e10_ok else 'FAIL'} | TGAT: {'OK' if tgat_ok else 'FAIL'}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("\u274c FastAPI did not start within 20 seconds.")

if token:
    print("\nStep 7: Creating ngrok tunnel...")
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print("\n" + "="*50)
    print("  E-COMMERCE ANOMALY DETECTION DEMO")
    print("  TGAT + E10 CatBoost Ensemble")
    print("="*50)
    print(f"  Dashboard: {public_url}")
    print(f"  API Docs:  {public_url}/docs")
    print(f"  Health:    {public_url}/health")
    print("="*50)

    print("\nStep 8: Smoke test (TGAT + E10 inference)...")
    sample = {"TransactionAmt": 999.99, "ProductCD": "W", "card1": 10409, "P_emaildomain": "anonymous.com"}
    try:
        resp = requests.post("http://127.0.0.1:8000/predict", json=sample, timeout=15)
        if resp.status_code == 200:
            pred = resp.json()
            print(f"\u2705 Smoke test passed!")
            print(f"   E10 prediction:    {pred['prediction']}")
            print(f"   E10 probability:   {pred['fraud_probability']}")
            print(f"   E10 risk level:    {pred['risk_level']}")
            tgat = pred.get('tgat', {})
            print(f"   TGAT enabled:      {tgat.get('enabled', False)}")
            print(f"   TGAT probability:  {tgat.get('probability', 'N/A')}")
            print(f"   Temporal decay tau:{tgat.get('temporal_decay_tau', 'N/A')}")
        else:
            print(f"\u274c Smoke test failed: HTTP {resp.status_code}")
    except Exception as e:
        print(f"\u274c Smoke test exception: {e}")
else:
    print("\n\u26a0 Skipping ngrok (no token). API is running locally on port 8000.")